In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()
zip_file = next(iter(uploaded))

with zipfile.ZipFile(zip_file, "r") as z:
    z.extractall("/content/olist_data")

print("Dataset extracted successfully!")
print(os.listdir("/content/olist_data"))

Saving archive.zip to archive.zip
Dataset extracted successfully!
['olist_order_reviews_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_customers_dataset.csv', 'olist_order_items_dataset.csv', 'olist_products_dataset.csv', 'olist_order_payments_dataset.csv']


In [ ]:
data_path = "/content/olist_data"

orders = pd.read_csv(
    f"{data_path}/olist_orders_dataset.csv"
)

customers = pd.read_csv(
    f"{data_path}/olist_customers_dataset.csv"
)

print("Orders shape:", orders.shape)
print("Customers shape:", customers.shape)

display(orders.head())
display(customers.head())

Orders shape: (99441, 8)
Customers shape: (99441, 5)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
# Convert order purchase timestamp to datetime
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

# Merge orders with customer data
merged_data = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

# Check the merged dataset
print("Merged dataset shape:", merged_data.shape)
print("Missing customer identities:",
      merged_data["customer_unique_id"].isna().sum())

display(
    merged_data[
        ["order_id", "customer_id",
         "customer_unique_id", "order_purchase_timestamp"]
    ].head()
)

Merged dataset shape: (99441, 9)
Missing customer identities: 0


,order_id,customer_id,customer_unique_id,order_purchase_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39


In [ ]:
# 1. Check order status
print("Order status count:")
print(merged_data["order_status"].value_counts())

# 2. Check duplicate orders
print("\nDuplicate order IDs:",
      merged_data["order_id"].duplicated().sum())

# 3. Check missing values
print("\nMissing values:")
print(merged_data[
    ["order_id", "customer_unique_id",
     "order_status", "order_purchase_timestamp"]
].isnull().sum())

# 4. Check purchase date range
print("\nPurchase date range:")
print(merged_data["order_purchase_timestamp"].min())
print(merged_data["order_purchase_timestamp"].max())

Order status count:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Duplicate order IDs: 0

Missing values:
order_id                    0
customer_unique_id          0
order_status                0
order_purchase_timestamp    0
dtype: int64

Purchase date range:
2016-09-04 21:15:19
2018-10-17 17:30:18


In [ ]:
z# Keep only delivered orders
delivered_data = merged_data[
    merged_data["order_status"] == "delivered"
].copy()

print("Total orders:", len(merged_data))
print("Delivered orders:", len(delivered_data))
print("Unique customers:",
      delivered_data["customer_unique_id"].nunique())

display(delivered_data.head())

Total orders: 99441
Delivered orders: 96478
Unique customers: 93358


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6


In [3]:
import os

print("Current directory:", os.getcwd())
print("Files available:", os.listdir("/content"))

Current directory: /content
Files available: ['.config', 'sample_data']
